# ComfyUI Colab runner for 3D-print timelapse video
Run the following cells in order on a GPU runtime. The setup keeps models and outputs in Google Drive so that they survive Colab restarts.

## 1. Mount Google Drive
Authorizes Drive and creates persistent locations for checkpoints, LoRAs, and rendered output.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

storage_root = Path('/content/drive/MyDrive/ComfyUI_Storage')
for directory in (
    storage_root / 'models' / 'checkpoints',
    storage_root / 'models' / 'loras',
    storage_root / 'output',
):
    directory.mkdir(parents=True, exist_ok=True)
    print(f'Ready: {directory}')


## 2. Clone ComfyUI and link persistent storage
This cell makes the temporary Colab checkout use Drive-backed model and output directories. It can safely be rerun after a runtime reset.

In [ ]:
%%bash
set -euo pipefail
if [ ! -d /content/ComfyUI/.git ]; then
  git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
else
  git -C /content/ComfyUI pull --ff-only
fi

DRIVE_ROOT=/content/drive/MyDrive/ComfyUI_Storage
for name in checkpoints loras; do
  target=$DRIVE_ROOT/models/$name
  link=/content/ComfyUI/models/$name
  rm -rf "$link"
  ln -s "$target" "$link"
done
rm -rf /content/ComfyUI/output
ln -s "$DRIVE_ROOT/output" /content/ComfyUI/output
ls -l /content/ComfyUI/models /content/ComfyUI | sed -n '1,80p'


## 3. Install dependencies and ComfyUI-Manager
The CUDA 12.1 wheels work on a T4 Colab runtime. ComfyUI-Manager helps install nodes required by a specific imported workflow.

In [ ]:
%%bash
set -euo pipefail
cd /content/ComfyUI
python -m pip install --upgrade pip
python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
python -m pip install xformers
python -m pip install -r requirements.txt
if [ ! -d custom_nodes/ComfyUI-Manager/.git ]; then
  git clone https://github.com/ltdrdata/ComfyUI-Manager custom_nodes/ComfyUI-Manager
else
  git -C custom_nodes/ComfyUI-Manager pull --ff-only
fi
python -m pip install -r custom_nodes/ComfyUI-Manager/requirements.txt || true


## 4. Launch ComfyUI through a public tunnel
The server and Cloudflare Quick Tunnel run in the background. Open the `trycloudflare.com` URL printed below. Treat it as a temporary public URL and stop the runtime when you are done.

In [ ]:
%%bash
set -euo pipefail
cd /content/ComfyUI
if ! command -v cloudflared >/dev/null 2>&1; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
  dpkg -i /tmp/cloudflared.deb
fi
pkill -f 'python main.py' || true
pkill -f 'cloudflared tunnel' || true
nohup python main.py --dont-print-server --listen > /content/comfyui.log 2>&1 &
sleep 8
nohup cloudflared tunnel --url http://127.0.0.1:8188 > /content/cloudflared.log 2>&1 &
sleep 5
grep -oE 'https://[-a-z0-9]+\.trycloudflare\.com' /content/cloudflared.log | tail -n 1 || (cat /content/cloudflared.log && false)
echo 'ComfyUI log: /content/comfyui.log'
